# SSIF_V3：Google Colab 中文實作教學（combined_data.csv 修正版）

研究資料：

```python
data_path = '/content/drive/MyDrive/00_SSIF/combined_data.csv'
```

`combined_data.csv` 的實際結構是 **每列一個完整事件**，欄位包含 `times`、`stids`、`intensity`、`epicenter_distance`、`variables`、`eq_info` 與 `source_file`。其中 `intensity` 是 `{station_id: [120 秒震度序列]}` 字典，不是每站一列的普通 sequence table。

本 Notebook 依序執行：掛載 Drive → clone 最新程式 → inspect CSV → 兩列小型轉換測試 → 完整轉換 → 輸出驗證 → 資料稽核與事件切分 → 快速訓練 → 正式訓練。


## 1. 掛載 Google Drive 並取得最新版程式


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
%cd /content
!rm -rf SSIF_V3
!git clone https://github.com/oceanicdayi/SSIF_V3.git
%cd /content/SSIF_V3
!git rev-parse HEAD
!python -m pip install -q -r requirements.txt


## 2. 設定資料與輸出路徑


In [ ]:
from pathlib import Path
import json
import shutil
import subprocess
import pandas as pd
import torch
from IPython.display import display

data_path = '/content/drive/MyDrive/00_SSIF/combined_data.csv'
DATA_CSV = Path(data_path)
REPO_ROOT = Path('/content/SSIF_V3')
WORK_ROOT = Path('/content/drive/MyDrive/00_SSIF/SSIF_V3_workspace')

TRAIN_DATA = WORK_ROOT / 'data' / 'training_archive_json'
EXTERNAL_DATA = WORK_ROOT / 'data' / 'external_evaluation_json'
PREPARED_DIR = WORK_ROOT / 'prepared' / 'split_v1'
MODEL_DIR = WORK_ROOT / 'models' / 'seed_20260728'
INFERENCE_DIR = WORK_ROOT / 'inference' / 'external_seed_20260728'
REPLAY_DIR = WORK_ROOT / 'replay'

WINDOWS = [10, 15, 20, 25, 30, 35, 40]
SEED = 20260728

for path in [TRAIN_DATA, EXTERNAL_DATA, PREPARED_DIR, MODEL_DIR, INFERENCE_DIR, REPLAY_DIR]:
    path.mkdir(parents=True, exist_ok=True)

assert DATA_CSV.is_file(), f'找不到研究資料：{DATA_CSV}'
print('CSV:', DATA_CSV)
print('CSV size (GB):', round(DATA_CSV.stat().st_size / 1024**3, 3))
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')


## 3. Inspect 實際 CSV schema


In [ ]:
inspect_cmd = [
    'python', 'csv_to_ssif_json.py', 'inspect',
    '--csv', str(DATA_CSV),
    '--rows', '2',
    '--horizon', '120',
]
inspect_result = subprocess.run(
    inspect_cmd, cwd=REPO_ROOT, check=True,
    text=True, capture_output=True,
)
schema = json.loads(inspect_result.stdout)
print(json.dumps(schema['detected'], ensure_ascii=False, indent=2))
print('Columns:', schema['columns'])

assert schema['detected']['layout'] == 'event_json', (
    '預期 combined_data.csv 為 event_json layout，實際為：'
    + str(schema['detected']['layout'])
)
assert {'eq_info', 'intensity'}.issubset(schema['columns'])
print('PASS: detected event_json layout')


## 4. 先用前兩列做端到端轉換測試

這一步不處理完整 728 MB 檔案。它會從原 CSV 複製前兩個事件，測試巢狀字典解析、事件 JSON 輸出與 120 秒資料驗證。


In [ ]:
SAMPLE_CSV = Path('/content/combined_data_sample_2rows.csv')
SAMPLE_OUT = Path('/content/ssif_csv_sample_events')

sample_df = pd.read_csv(DATA_CSV, nrows=2, low_memory=False)
sample_df.to_csv(SAMPLE_CSV, index=False)
shutil.rmtree(SAMPLE_OUT, ignore_errors=True)

sample_convert_cmd = [
    'python', 'csv_to_ssif_json.py', 'convert',
    '--csv', str(SAMPLE_CSV),
    '--output-dir', str(SAMPLE_OUT),
    '--layout', 'event_json',
    '--horizon', '120',
    '--chunk-size', '1',
    '--overwrite',
]
result = subprocess.run(
    sample_convert_cmd, cwd=REPO_ROOT, check=True,
    text=True, capture_output=True,
)
sample_summary = json.loads(result.stdout)
print(json.dumps(sample_summary, ensure_ascii=False, indent=2))

sample_validate_cmd = [
    'python', 'csv_to_ssif_json.py', 'validate',
    '--data-dir', str(SAMPLE_OUT),
    '--horizon', '120',
]
validated = subprocess.run(
    sample_validate_cmd, cwd=REPO_ROOT, check=True,
    text=True, capture_output=True,
)
sample_validation = json.loads(validated.stdout)
print(json.dumps(sample_validation, ensure_ascii=False, indent=2))

assert sample_summary['n_events'] == 2
assert sample_validation['n_errors'] == 0
assert sample_validation['n_event_json'] == 2
print('PASS: two-row conversion and validation')


## 5. 檢查一個轉換後事件


In [ ]:
sample_event_files = sorted(SAMPLE_OUT.glob('event_*.json'))
assert sample_event_files
with sample_event_files[0].open(encoding='utf-8') as f:
    sample_event = json.load(f)

print('Event file:', sample_event_files[0].name)
print('eq_info:', sample_event['eq_info'])
print('stations:', len(sample_event['intensity']))
first_station = next(iter(sample_event['intensity']))
print('first station:', first_station)
print('sequence length:', len(sample_event['intensity'][first_station]))
print('first 20 values:', sample_event['intensity'][first_station][:20])

assert isinstance(sample_event['eq_info'], dict)
assert isinstance(sample_event['intensity'], dict)
assert len(sample_event['intensity'][first_station]) == 120


## 6. 執行完整 CSV 轉換

先確認第 3–5 節全部 PASS，再把 `RUN_FULL_CONVERSION` 改為 `True`。`--overwrite` 會先清除舊的部分輸出，避免先前失敗的轉換殘留。轉換器預設遇到第一個錯誤就停止，避免靜默遺失事件。


In [ ]:
RUN_FULL_CONVERSION = False

if RUN_FULL_CONVERSION:
    full_convert_cmd = [
        'python', 'csv_to_ssif_json.py', 'convert',
        '--csv', str(DATA_CSV),
        '--output-dir', str(TRAIN_DATA),
        '--layout', 'event_json',
        '--horizon', '120',
        '--chunk-size', '100',
        '--duplicate-policy', 'error',
        '--row-error-policy', 'error',
        '--overwrite',
    ]
    print('Running full conversion...')
    completed = subprocess.run(
        full_convert_cmd, cwd=REPO_ROOT, check=True,
        text=True, capture_output=True,
    )
    conversion_summary = json.loads(completed.stdout)
    print(json.dumps(conversion_summary, ensure_ascii=False, indent=2))
else:
    print('尚未執行完整轉換。先完成兩列測試，再設定 RUN_FULL_CONVERSION=True。')


## 7. 驗證完整輸出，並與 CSV 列數核對


In [ ]:
RUN_FULL_VALIDATION = False

if RUN_FULL_VALIDATION:
    validate_cmd = [
        'python', 'csv_to_ssif_json.py', 'validate',
        '--data-dir', str(TRAIN_DATA),
        '--horizon', '120',
    ]
    checked = subprocess.run(
        validate_cmd, cwd=REPO_ROOT, check=True,
        text=True, capture_output=True,
    )
    archive_validation = json.loads(checked.stdout)
    print(json.dumps(archive_validation, ensure_ascii=False, indent=2))

    index_df = pd.read_csv(TRAIN_DATA / 'event_index.csv')
    with open(TRAIN_DATA / 'conversion_summary.json', encoding='utf-8') as f:
        conversion_summary = json.load(f)

    csv_rows = sum(len(chunk) for chunk in pd.read_csv(
        DATA_CSV, usecols=['eq_info'], chunksize=500, low_memory=False
    ))
    print('CSV event rows:', csv_rows)
    print('Converted JSON:', archive_validation['n_event_json'])
    print('Index rows:', len(index_df))
    print('Row errors:', conversion_summary['counters'].get('row_errors', 0))

    assert archive_validation['n_errors'] == 0
    assert archive_validation['n_event_json'] == csv_rows
    assert len(index_df) == csv_rows
    assert conversion_summary['counters'].get('row_errors', 0) == 0
    print('PASS: full archive count and schema validation')
else:
    print('完整轉換完成後，設定 RUN_FULL_VALIDATION=True。')


## 8. 執行程式內建 smoke tests


In [ ]:
!python smoke_test_csv_conversion.py
!python smoke_test_pipeline_v3.py


## 9. 資料稽核與事件層級四集合切分


In [ ]:
RUN_AUDIT_SPLIT = False

if RUN_AUDIT_SPLIT:
    audit_cmd = [
        'python', 'prepare_ssif_dataset.py', 'audit-split',
        '--data-dir', str(TRAIN_DATA),
        '--output-dir', str(PREPARED_DIR),
        '--windows', *map(str, WINDOWS),
        '--label-horizon', '120',
        '--min-label-valid-fraction', '0.80',
        '--min-window-valid-fraction', '0.80',
        '--train-ratio', '0.70',
        '--validation-ratio', '0.10',
        '--calibration-ratio', '0.10',
        '--test-ratio', '0.10',
        '--split-candidates', '5000',
        '--seed', str(SEED),
    ]
    subprocess.run(audit_cmd, cwd=REPO_ROOT, check=True)
else:
    print('完整轉換與驗證通過後，再設定 RUN_AUDIT_SPLIT=True。')


## 10. 查看稽核與切分結果


In [ ]:
if (PREPARED_DIR / 'audit_summary.json').is_file():
    with open(PREPARED_DIR / 'audit_summary.json', encoding='utf-8') as f:
        audit_summary = json.load(f)
    print(json.dumps(audit_summary, ensure_ascii=False, indent=2))

    event_split = pd.read_csv(PREPARED_DIR / 'event_split.csv')
    display(event_split['split'].value_counts().rename_axis('split').to_frame('events'))
    display(event_split.head())
else:
    print('尚未產生 audit outputs。')


## 11. EW10 一個 epoch 快速訓練


In [ ]:
RUN_QUICK_TRAIN = False
QUICK_MODEL_DIR = WORK_ROOT / 'models' / 'quick_EW10'

if RUN_QUICK_TRAIN:
    quick_cmd = [
        'python', 'train_ssif_v3.py', 'train-all',
        '--data-dir', str(TRAIN_DATA),
        '--split-manifest', str(PREPARED_DIR / 'split_manifest.json'),
        '--output-dir', str(QUICK_MODEL_DIR),
        '--windows', '10',
        '--label-horizon', '120',
        '--cohort', 'common',
        '--epochs', '1',
        '--batch-size', '16',
        '--eval-batch-size', '64',
        '--lr', '3e-4',
        '--seed', str(SEED),
        '--window-seed-mode', 'same',
        '--workers', '2',
    ]
    if torch.cuda.is_available():
        quick_cmd.append('--amp')
    subprocess.run(quick_cmd, cwd=REPO_ROOT, check=True)
    assert (QUICK_MODEL_DIR / 'EW10' / 'best.pt').is_file()
    print('PASS: EW10 quick training')
else:
    print('audit-split 通過後，再設定 RUN_QUICK_TRAIN=True。')


## 12. 正式訓練 EW10–EW40


In [ ]:
RUN_FULL_TRAIN = False

if RUN_FULL_TRAIN:
    train_cmd = [
        'python', 'train_ssif_v3.py', 'train-all',
        '--data-dir', str(TRAIN_DATA),
        '--split-manifest', str(PREPARED_DIR / 'split_manifest.json'),
        '--output-dir', str(MODEL_DIR),
        '--windows', *map(str, WINDOWS),
        '--label-horizon', '120',
        '--cohort', 'common',
        '--epochs', '30',
        '--batch-size', '16',
        '--eval-batch-size', '64',
        '--lr', '3e-4',
        '--weight-decay', '1e-2',
        '--warmup-ratio', '0.10',
        '--min-precision', '0.90',
        '--seed', str(SEED),
        '--window-seed-mode', 'same',
        '--patience', '6',
        '--workers', '2',
    ]
    if torch.cuda.is_available():
        train_cmd.append('--amp')
    subprocess.run(train_cmd, cwd=REPO_ROOT, check=True)
else:
    print('EW10 quick training 通過後，再設定 RUN_FULL_TRAIN=True。')


## 13. 研究資料完整性原則

1. `combined_data.csv` 只用來建立 train、validation、calibration 與 internal test。
2. external evaluation 必須是未參與模型選擇與 threshold calibration 的獨立事件。
3. 正式訓練前保存 `conversion_summary.json`、`event_index.csv`、`audit_summary.json`、`split_manifest.json` 與 Git commit SHA。
4. 不可使用 `--row-error-policy skip` 產生正式論文資料；該模式只適合診斷。
5. 若資料版本改變，建立新的 prepared/model 版本目錄，不覆蓋已凍結的實驗。
